# Part B: Dataset Understanding and Preparation

In [42]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold, StratifiedKFold, LeaveOneOut, TimeSeriesSplit, cross_val_score
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV

In [15]:
df = pd.read_excel(r"C:\Users\tanaa\Downloads\Advanced_Regression_HousePrice_Dataset_3800.xlsx")
df.head()

,property_id,sale_date,area_sqft,bedrooms,bathrooms,location_score,property_age,distance_city_km,near_school,near_metro,crime_rate_index,house_price_inr
0,200001,2014-01-01,2181,6,4,8.1,21,3.8,0,0,4.84,35154898
1,200002,2019-12-01,2383,5,4,5.3,28,10.9,1,1,2.89,26710893
2,200003,2016-10-01,1047,3,3,5.9,7,27.5,0,1,4.04,11216242
3,200004,2013-03-01,1753,4,3,7.0,27,12.1,0,0,3.28,21984310
4,200005,2013-07-01,1728,4,4,10.0,32,1.4,0,1,3.84,25080429


## Part B: Dataset Understanding & Preparation

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3800 entries, 0 to 3799
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   property_id       3800 non-null   int64         
 1   sale_date         3800 non-null   datetime64[ns]
 2   area_sqft         3800 non-null   int64         
 3   bedrooms          3800 non-null   int64         
 4   bathrooms         3800 non-null   int64         
 5   location_score    3800 non-null   float64       
 6   property_age      3800 non-null   int64         
 7   distance_city_km  3800 non-null   float64       
 8   near_school       3800 non-null   int64         
 9   near_metro        3800 non-null   int64         
 10  crime_rate_index  3800 non-null   float64       
 11  house_price_inr   3800 non-null   int64         
dtypes: datetime64[ns](1), float64(3), int64(8)
memory usage: 356.4 KB


In [17]:
df.shape

(3800, 12)

In [18]:
df.describe()

,property_id,sale_date,area_sqft,bedrooms,bathrooms,location_score,property_age,distance_city_km,near_school,near_metro,crime_rate_index,house_price_inr
count,3800.00000,3800,3800.000000,3800.000000,3800.000000,3800.000000,3800.000000,3800.000000,3800.000000,3800.000000,3800.000000,3.800000e+03
mean,201900.50000,2016-11-27 23:45:13.263158016,1716.925526,3.428158,2.916316,6.502237,22.537105,13.085132,0.548421,0.472895,4.242911,2.071940e+07
min,200001.00000,2010-01-01 00:00:00,500.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.500000,1.506126e+06
25%,200950.75000,2013-06-01 00:00:00,1322.000000,2.000000,2.000000,5.300000,14.000000,8.500000,0.000000,0.000000,2.810000,1.446895e+07
50%,201900.50000,2016-12-01 00:00:00,1700.500000,3.000000,3.000000,6.500000,20.000000,13.000000,1.000000,0.000000,4.220000,1.989180e+07
75%,202850.25000,2020-05-01 00:00:00,2105.000000,4.000000,4.000000,7.700000,29.000000,17.500000,1.000000,1.000000,5.650000,2.596062e+07
max,203800.00000,2023-12-01 00:00:00,3776.000000,7.000000,6.000000,10.000000,80.000000,38.700000,1.000000,1.000000,12.000000,5.930315e+07
std,1097.10984,NaN,582.996559,1.356682,1.133540,1.766945,12.325740,6.537425,0.497715,0.499330,2.045371,8.707465e+06


In [19]:
df.isna().sum()

property_id         0
sale_date           0
area_sqft           0
bedrooms            0
bathrooms           0
location_score      0
property_age        0
distance_city_km    0
near_school         0
near_metro          0
crime_rate_index    0
house_price_inr     0
dtype: int64

In [20]:
df.duplicated()

0       False
1       False
2       False
3       False
4       False
        ...  
3795    False
3796    False
3797    False
3798    False
3799    False
Length: 3800, dtype: bool

## Feature Extraction

In [21]:
df["sale_date"] = pd.to_datetime(df["sale_date"])
df["date"] = df["sale_date"].dt.day
df["month"] = df["sale_date"].dt.month
df["year"] = df["sale_date"].dt.year

df.drop("sale_date", axis = 1, inplace=True)

## Train Test Split

In [22]:
from sklearn.model_selection import train_test_split
x = df.drop("house_price_inr", axis=1)
y = df["house_price_inr"]

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.3,
    random_state=32
)

## Scaling

In [23]:
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

# Part C: Regularized Linear Models

In [26]:
ridge = RidgeCV(alphas=[0.01, 0.1, 1, 10, 100], cv=5)
ridge.fit(x_train_scaled, y_train)
print("Ridge best alpha:", ridge.alpha_)

lasso = LassoCV(alphas=[0.01, 0.1, 1, 10, 100], cv=5)
lasso.fit(x_train_scaled, y_train)
print("Lasso best alpha:", lasso.alpha_)

Ridge best alpha: 1.0
Lasso best alpha: 100.0


In [27]:
ridge_train_pred = ridge.predict(x_train_scaled)
ridge_test_pred = ridge.predict(scaler.transform(x_test))

lasso_train_pred = lasso.predict(x_train_scaled)
lasso_test_pred = lasso.predict(scaler.transform(x_test))


print("Ridge Regression:-")
print("Training RMSE:", np.sqrt(mean_squared_error(y_train, ridge_train_pred)))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, ridge_test_pred)))
print("r2_score:", r2_score(y_test, ridge_test_pred))


print("\nLasso Regression:-")
print("Training RMSE:", np.sqrt(mean_squared_error(y_train, lasso_train_pred)))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, lasso_test_pred)))
print("R2_score:", r2_score(y_test, lasso_test_pred))

Ridge Regression:-
Training RMSE: 2513950.8536799694
Test RMSE: 2444885.596068199
r2_score: 0.918839985041814

Lasso Regression:-
Training RMSE: 2513945.880619565
Test RMSE: 2445027.2036317997
R2_score: 0.9188305832073628


In [28]:
coefficient = pd.DataFrame({
    "Features":x_train.columns,
    "Ridge_coefficient":ridge.coef_,
    "Lasso_coefficient":lasso.coef_
})

print(coefficient.round(3))
print("Ridge zero coefficient:", np.sum(np.isclose(ridge.coef_, 0)))
print("Lasso zero coefficient:", np.sum(np.isclose(lasso.coef_, 0)))

            Features  Ridge_coefficient  Lasso_coefficient
0        property_id         -10546.365         -10460.773
1          area_sqft        7072501.905        7080902.243
2           bedrooms         329313.471         322539.463
3          bathrooms         189230.145         188942.336
4     location_score        3724825.020        3727192.492
5       property_age        -663281.218        -663354.143
6   distance_city_km         -21591.360         -19958.541
7        near_school         -18504.017         -18423.288
8         near_metro          34533.043          34577.655
9   crime_rate_index        -198659.656        -198498.421
10              date              0.000              0.000
11             month           4172.270           3938.765
12              year         310603.648         310660.051
Ridge zero coefficient: 1
Lasso zero coefficient: 1


# Part D: Cross-Validation Strategies

In [30]:
model = make_pipeline(StandardScaler(), Ridge(alpha=1))
kfold = KFold(n_splits=5, shuffle=True, random_state=32)
scores = cross_val_score(model, x_train, y_train, cv=kfold, scoring="neg_mean_squared_error")
print("k-fold RMSE:", np.sqrt(-scores).mean())

k-fold RMSE: 2528879.673452137


In [31]:
loo = LeaveOneOut()
scores = cross_val_score(model, x_train, y_train, cv=loo, scoring="neg_mean_squared_error", n_jobs=-1)
print("LOOCV RMSE:", np.sqrt(-scores).mean())

LOOCV RMSE: 1929860.6584261672


In [32]:
x_time = x_train.sort_values(["year", "month", "date"])
y_time = y_train.loc[x_time.index]

tscv = TimeSeriesSplit(n_splits=5)

scores = cross_val_score(model, x_time, y_time,cv=tscv,
                         scoring="neg_mean_squared_error")

print("Time Series RMSE:", np.sqrt(-scores).mean())

Time Series RMSE: 2521371.4123724075


# Part E: Tree-Based Regression Models

In [34]:
dt = DecisionTreeRegressor(random_state=32)
dt.fit(x_train, y_train)

y_pred_dt = dt.predict(x_test)

print("Decision tree RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_dt)))
print("R2_score:", r2_score(y_test, y_pred_dt))

Decision tree RMSE: 3286658.2956653354
R2_score: 0.8533324144237239


In [35]:
dt_tuned = DecisionTreeRegressor(
    max_depth=10, min_samples_leaf=5, min_samples_split=10, random_state=32
)
dt_tuned.fit(x_train, y_train)
pred_dt_tuned = dt_tuned.predict(x_test)

print("Tuned decision tree RMSE:", np.sqrt(mean_squared_error(y_test, pred_dt_tuned)))
print("R2_score:", r2_score(y_test, y_pred_dt))

Tuned decision tree RMSE: 2868740.2829214227
R2_score: 0.8533324144237239


In [37]:
rf = RandomForestRegressor(
    n_estimators=100, max_depth=10, random_state=32, n_jobs=-1
)

rf.fit(x_train, y_train)
pred_rf = rf.predict(x_test)

print("Tuned decision tree RMSE:", np.sqrt(mean_squared_error(y_test, pred_rf)))
print("R2_score:", r2_score(y_test, pred_rf))

Tuned decision tree RMSE: 2311358.053230028
R2_score: 0.9274630168085167


In [38]:
results = {
    "Decision Tree": r2_score(y_test, y_pred_dt),
    "Tuned Decision Tree": r2_score(y_test, pred_dt_tuned),
    "Random Forest": r2_score(y_test, pred_rf)
}

print(pd.Series(results, name="R2 Score"))

Decision Tree          0.853332
Tuned Decision Tree    0.888260
Random Forest          0.927463
Name: R2 Score, dtype: float64


# Part F: Support Vector Regression

In [40]:
svr_linear = make_pipeline(
    StandardScaler(),
    SVR(kernel="linear")
)

svr_linear.fit(x_train, y_train)
y_pred_linear = svr_linear.predict(x_test)

print("Linear SVR RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_linear)))
print("Linear SVR R2:", r2_score(y_test, y_pred_linear))

Linear SVR RMSE: 8630656.146064045
Linear SVR R2: -0.011376045245617528


In [41]:
svr_rbf = make_pipeline(
    StandardScaler(),
    SVR(kernel="rbf")
)

svr_rbf.fit(x_train, y_train)
y_pred_rbf = svr_rbf.predict(x_test)

print("RBF SVR RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rbf)))
print("RBF SVR R2:", r2_score(y_test, y_pred_rbf))

RBF SVR RMSE: 8634648.70230278
RBF SVR R2: -0.012311990244010973


In [43]:
params = {
    "svr__C": [1, 10, 100],
    "svr__gamma": ["scale", 0.01, 0.1],
    "svr__epsilon": [0.1, 0.2, 0.5]
}

grid = GridSearchCV(
    svr_rbf,
    params,
    cv=3,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

grid.fit(x_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best RMSE:", np.sqrt(-grid.best_score_))

Best Parameters: {'svr__C': 100, 'svr__epsilon': 0.1, 'svr__gamma': 'scale'}
Best RMSE: 8806168.233315011


In [44]:
results = { "Ridge": r2_score(y_test, ridge.predict(x_test_scaled)), "Decision Tree": r2_score(y_test, y_pred_dt), "Random Forest": r2_score(y_test, pred_rf), "Linear SVR": r2_score(y_test, y_pred_linear), "RBF SVR Tuned": r2_score(y_test, grid.best_estimator_.predict(x_test)) }

print(pd.Series(results, name="R2 Score").sort_values(ascending=False))

Random Forest    0.927463
Ridge            0.918840
Decision Tree    0.853332
RBF SVR Tuned   -0.010433
Linear SVR      -0.011376
Name: R2 Score, dtype: float64


In [45]:
print("MSE:", mean_squared_error(y_test, pred_rf))
print("MAE:", mean_absolute_error(y_test, pred_rf))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_rf)))
print("R2 Score:", r2_score(y_test, pred_rf))

MSE: 5342376050231.306
MAE: 1759979.4454222736
RMSE: 2311358.053230028
R2 Score: 0.9274630168085167


# Final Conclusion: House Price Prediction

In this project, different machine learning regression models were studied and compared to predict house prices. The models included Ridge Regression, Lasso Regression, Decision Tree Regression, Random Forest Regression, and Support Vector Regression (SVR).

Among the tested models, **Random Forest performed best**, achieving an R² score of **0.927463**, followed by Ridge Regression with an R² score of **0.918840**. Decision Tree achieved an R² score of **0.853332**, while Linear SVR and RBF SVR showed poor predictive performance with negative R² scores.

Regularization techniques such as Ridge and Lasso help control model complexity and reduce overfitting. Cross-validation helps evaluate model stability and select suitable hyperparameters. Tree-based models can capture complex, nonlinear relationships in house-price data, while SVR may require better feature scaling and hyperparameter tuning.

Overfitting occurs when a model performs well on training data but poorly on unseen data. Underfitting occurs when a model performs poorly on both training and testing data. Comparing training and testing metrics helps identify these problems.

From a business perspective, house-price prediction can support property valuation, pricing decisions, and real-estate market analysis. Random Forest is a promising candidate based on the reported results, but MAE, RMSE, and validation performance should also be considered before deployment.

**Overall, Random Forest was the best-performing model among the tested models, followed by Ridge Regression.** The results demonstrate the importance of regularization, cross-validation, model comparison, and proper evaluation when developing a reliable house-price prediction system.